In [1]:
import pandas as pd
import numpy as np

# 1. Generate a deliberately messy dataset
np.random.seed(42)

# Creating messy data with intentional errors, inconsistent text, and extreme outliers
data = {
    'CustomerID': ['C001', 'C002', 'C003', 'C004', 'C005', 'C005', 'C007', 'C008', 'C009', 'C010', 'C011', 'C012', 'C013', 'C014', 'C015'] * 2,
    'Age': [25, np.nan, 30, 22, 250, 250, 45, 33, np.nan, 29, 31, 85, 27, 40, 22] * 2,
    'Gender': ['Male', 'Female', 'M', 'F', 'male', 'male', 'Female', 'Fem', 'M', 'Female', 'Male', 'F', 'm', 'Female', 'Male'] * 2,
    'Income': [50000, 60000, 55000, 45000, 1500000, 1500000, np.nan, 72000, 58000, 61000, 59000, 48000, 52000, np.nan, 46000] * 2,
    'Signup_Date': ['2023-01-15', '15/01/2023', '2023-02-20', '2023-03-05', '04-05-2023', '04-05-2023', '2023-06-12', '2023-07-22', '2023-08-30', '09/09/2023', '2023-10-10', '2023-11-11', '12-12-2023', '2023-12-25', '2023-12-31'] * 2
}

df = pd.DataFrame(data)

# Adding completely empty rows to make it extra messy
df.loc[30] = [np.nan, np.nan, np.nan, np.nan, np.nan]
df.loc[31] = [np.nan, np.nan, np.nan, np.nan, np.nan]

# 2. The Initial Data Quality Report
print("--- DATA QUALITY REPORT (BEFORE CLEANING) ---")
print(f"Total Rows: {len(df)}")
print(f"Total Duplicates: {df.duplicated().sum()}")
print("\n--- NULL VALUES PER COLUMN ---")
print(df.isnull().sum())
print("\n--- DATA TYPES ---")
print(df.dtypes)
print("\n--- VALUE RANGES (Checking for Anomalies) ---")
display(df.describe())

--- DATA QUALITY REPORT (BEFORE CLEANING) ---
Total Rows: 32
Total Duplicates: 17

--- NULL VALUES PER COLUMN ---
CustomerID     2
Age            6
Gender         2
Income         6
Signup_Date    2
dtype: int64

--- DATA TYPES ---
CustomerID      object
Age            float64
Gender          object
Income         float64
Signup_Date     object
dtype: object

--- VALUE RANGES (Checking for Anomalies) ---


,Age,Income
count,26.000000,2.600000e+01
mean,68.384615,2.773846e+05
std,80.582418,5.316973e+05
min,22.000000,4.500000e+04
25%,27.000000,5.000000e+04
50%,31.000000,5.800000e+04
75%,45.000000,6.100000e+04
max,250.000000,1.500000e+06


Data Quality Report: Initial Assessment
Upon loading the raw dataset, several critical data quality issues were immediately identified that require cleaning:

Duplicates: There are 17 identical duplicate rows that need to be removed.

Missing Data (Nulls): The Age and Income columns contain missing numeric values, and two completely blank rows exist at the bottom of the dataset.

Standardization Issues: The Gender column contains inconsistent capitalization and abbreviations (e.g., "Male", "M", "male", "m"). Furthermore, the Signup_Date column has a mix of formatting styles (YYYY-MM-DD vs. DD/MM/YYYY) and is currently being read as an "object" (string) rather than a proper datetime data type.

Outliers/Anomalies: The descriptive statistics reveal extreme, impossible outliers. The maximum Age is recorded as 250 years old, and the maximum Income sits at $1,500,000, which heavily skews the dataset's mean

In [2]:
# 1. Make a copy of the raw dataset to work on
cleaned_df = df.copy()

# 2. Remove completely blank rows and drop duplicate rows
cleaned_df.dropna(how='all', inplace=True)
cleaned_df.drop_duplicates(inplace=True)

# 3. Standardize the 'Gender' column values
gender_mapping = {
    'Male': 'Male', 'male': 'Male', 'M': 'Male', 'm': 'Male',
    'Female': 'Female', 'Fem': 'Female', 'F': 'Female'
}
cleaned_df['Gender'] = cleaned_df['Gender'].map(gender_mapping)

# 4. Handle Outliers in 'Age' and 'Income' before imputing missing values
# Replace impossible age (> 100) and extreme income (> $200k) with NaN
cleaned_df.loc[cleaned_df['Age'] > 100, 'Age'] = np.nan
cleaned_df.loc[cleaned_df['Income'] > 200000, 'Income'] = np.nan

# 5. Handle Missing Data using Median Imputation
cleaned_df['Age'] = cleaned_df['Age'].fillna(cleaned_df['Age'].median())
cleaned_df['Income'] = cleaned_df['Income'].fillna(cleaned_df['Income'].median())

# 6. Fix Data Types & Standardize Dates
cleaned_df['Signup_Date'] = pd.to_datetime(cleaned_df['Signup_Date'], errors='coerce', dayfirst=True)
cleaned_df['Age'] = cleaned_df['Age'].astype(int)

# 7. Save the cleaned dataset to a new CSV file
cleaned_df.to_csv('cleaned_customer_data.csv', index=False)

# 8. Display the "Before vs. After" Summary Table
summary_data = {
    'Metric': ['Total Rows', 'Duplicate Rows', 'Null Values (Age)', 'Null Values (Income)', 'Max Age Value', 'Max Income Value', 'Date Data Type'],
    'Before Cleaning': [len(df), df.duplicated().sum(), df['Age'].isnull().sum(), df['Income'].isnull().sum(), f"{df['Age'].max()} yrs", f"${df['Income'].max():,.0f}", str(df['Signup_Date'].dtype)],
    'After Cleaning': [len(cleaned_df), cleaned_df.duplicated().sum(), cleaned_df['Age'].isnull().sum(), cleaned_df['Income'].isnull().sum(), f"{cleaned_df['Age'].max()} yrs", f"${cleaned_df['Income'].max():,.0f}", str(cleaned_df['Signup_Date'].dtype)]
}

summary_df = pd.DataFrame(summary_data)
print("--- BEFORE vs. AFTER DATA CLEANING SUMMARY ---")
display(summary_df)

--- BEFORE vs. AFTER DATA CLEANING SUMMARY ---


/tmp/ipykernel_1173/1295022606.py:25: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  cleaned_df['Signup_Date'] = pd.to_datetime(cleaned_df['Signup_Date'], errors='coerce', dayfirst=True)


,Metric,Before Cleaning,After Cleaning
0,Total Rows,32,14
1,Duplicate Rows,17,0
2,Null Values (Age),6,0
3,Null Values (Income),6,0
4,Max Age Value,250.0 yrs,85 yrs
5,Max Income Value,"$1,500,000","$72,000"
6,Date Data Type,object,datetime64[ns]


Data Cleaning Methodology & Decision Log

Duplicate & Blank Removal: Completely uninformative blank rows and duplicate entries were deleted first to ensure data integrity and prevent double-counting.

Text Standardization: Inconsistent string representations in the Gender column were mapped into two clean categories (Male and Female) using a dictionary lookup.

Outlier Treatment & Imputation: Extreme anomalies (such as an age of 250 years and income of $1.5M) were identified using domain knowledge and converted to missing values (NaN). Missing values in both Age and Income were subsequently filled using median imputation to protect against skewed distributions.

Format Conversion: All date strings were parsed into standard datetime64 format, ensuring consistent chronological sorting for future analysis.

Export: The cleaned, analysis-ready dataset was exported to cleaned_customer_data.csv.